# 04. Changing Photos — Remove Background · 4x Resolution · Depth

> In 01~03, you put in a photo and got **numbers (boxes · probabilities)** out. These three models take a photo and give back **a photo (a grid of numbers of the same size)**.
> So post-processing becomes "turning the grid of numbers back into a photo".

### What you will learn
1. **Normalization** — why we subtract the mean and divide by the standard deviation
2. Stretching an output grid (mask · depth) to 0~1 and turning it into a photo, **alpha blending**
3. A model with **two** inputs (4x resolution)
4. What relative depth means, and its limits
5. Combining the results — background blur, portrait mode

| Model | Input | Preprocessing | Output |
|---|---|---|---|
| U2Net (ONNX) | 320 x 320 RGB | ÷255 → (x − mean) ÷ std | mask, 1 channel — bigger value = more in front (the object) |
| SR-1032 | small photo 480 x 270 + enlarged photo 1920 x 1080 | none (0~255) | 1920 x 1080, values 0~1 |
| Depth Anything V2 Small | 518 x 518 RGB | same normalization as U2Net | relative depth (bigger = closer) |

## 1. Setup — normalization values
Many photo models were pre-trained on a photo collection called **ImageNet**. We must scale the input with the per-color mean · standard deviation of those photos, so it falls in the same range the model learned.

$$x' = \frac{x / 255 - \text{mean}}{\text{std}} \qquad \text{mean} = (0.485,\ 0.456,\ 0.406),\ \text{std} = (0.229,\ 0.224,\ 0.225) \ (\text{R, G, B})$$

The result is roughly between −2 and +2.

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us); this notebook is in notebooks/en/
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

def pick(*prefer):
    """Pick the first device in the list that this PC has."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

DEVICE = pick("GPU", "CPU")                        # models that handle big photos — GPU
G = MODELS / "gan"
MEAN = np.array([0.485, 0.456, 0.406], np.float32)  # ImageNet mean · standard deviation (RGB)
STD = np.array([0.229, 0.224, 0.225], np.float32)

def normalized(bgr, size):
    rgb = cv2.cvtColor(cv2.resize(bgr, (size, size)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return ((rgb - MEAN) / STD).transpose(2, 0, 1)[None].astype(np.float32)
print("Device:", DEVICE)

## 2. Photo

In [ ]:
PHOTO = "sample.jpg"          # photo to use when there is no webcam (in the same folder as this notebook)

def show(bgr, width=720):
    """Show an OpenCV photo (BGR) inside the notebook."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """Take one frame from the webcam. If there is no webcam, use the PHOTO file."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # some webcams give dark first frames
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"No webcam and no {PHOTO} — put a photo file name in PHOTO"
        print("No webcam, so using the file:", PHOTO)
    return frame

frame = grab()
print("Photo size (height, width, channels):", frame.shape)
show(frame)

## 3. Remove the background — U2Net
U2Net finds the **thing that stands out (salient object)** in a photo and returns a grid that is "1 for the front, 0 for the back". It is an ONNX file, but OpenVINO opens it directly.

**Post-processing**
1. Stretch the output to 0~1 within the photo (min-max normalization): $m' = \frac{m - \min}{\max - \min}$
2. Resize it to the original photo size.
3. **Alpha blending**: $\text{result} = \text{photo} \times m + \text{background} \times (1 - m)$ — where m is 1 you get the photo, where it is 0 you get the background.

In [ ]:
u2 = core.compile_model(core.read_model(G / "u2net.onnx"), DEVICE)
print("Number of outputs:", len(u2.outputs), "— the first output is the finest mask")

h, w = frame.shape[:2]
out = u2(normalized(frame, 320))[u2.output(0)]
mask = out.squeeze().astype(np.float32)
mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)   # ① stretch to 0~1
mask = cv2.resize(mask, (w, h))[..., None]                      # ② back to original size, add one channel axis (so we can multiply)
cut = (frame * mask + 255 * (1 - mask)).astype(np.uint8)        # ③ make the background white
show(np.hstack([frame, cv2.cvtColor((mask * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR), cut]))

In [ ]:
# Use it: blur only the background — put a 'blurred copy of the photo' where the background is
blur = cv2.GaussianBlur(frame, (0, 0), 12)
soft = (frame * mask + blur * (1 - mask)).astype(np.uint8)
hard = np.where(mask > 0.5, frame, blur).astype(np.uint8)      # a hard cut at 0.5 gives jagged edges
print("Left: soft blend | Right: hard cut at 0.5")
show(np.hstack([soft, hard]))

## 4. 4x resolution — a model with **two** inputs
SR-1032 takes a small photo (480 x 270) **together with** the same photo simply enlarged (1920 x 1080).
The model only creates the "details to add" to the enlarged photo (residual learning). The input size is fixed, so for a photo with a different aspect ratio we fill the empty area and cut it off later.

When there are several inputs, pass them as a `{name: array}` dictionary.

In [ ]:
sr = core.compile_model(G / "single-image-super-resolution-1032.xml", DEVICE)
print("Inputs:", [(i.any_name, tuple(i.shape)) for i in sr.inputs])

def super_res(bgr):
    h, w = bgr.shape[:2]
    s = min(480 / w, 270 / h)
    nw, nh = max(1, int(w * s)), max(1, int(h * s))
    small = cv2.copyMakeBorder(cv2.resize(bgr, (nw, nh)), 0, 270 - nh, 0, 480 - nw, cv2.BORDER_REPLICATE)
    big = cv2.resize(small, (1920, 1080), interpolation=cv2.INTER_CUBIC)
    feed = {sr.inputs[0].any_name: small.transpose(2, 0, 1)[None].astype(np.float32),
            sr.inputs[1].any_name: big.transpose(2, 0, 1)[None].astype(np.float32)}
    out = sr(feed)[sr.output(0)].squeeze().transpose(1, 2, 0)   # CHW → HWC
    out = (out * 255).clip(0, 255).astype(np.uint8)             # 0~1 → 0~255
    return out[: nh * 4, : nw * 4], small[:nh, :nw]             # cut off the filled area

t0 = time.perf_counter()
big4, small = super_res(frame)
print(f"{small.shape[1]}x{small.shape[0]} → {big4.shape[1]}x{big4.shape[0]}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")
plain = cv2.resize(small, (big4.shape[1], big4.shape[0]), interpolation=cv2.INTER_CUBIC)
cy, cx = big4.shape[0] // 2, big4.shape[1] // 2          # compare the center part up close
crop = lambda im: im[cy - 120: cy + 120, cx - 160: cx + 160]
print("Left: simple enlarge (bicubic) | Right: model")
show(np.hstack([crop(plain), crop(big4)]))

## 5. Depth — near and far from a single photo
Depth Anything V2 guesses how close each pixel is, from just one photo.
- **Relative depth**: it knows "this is closer than that", but **not how many meters**. Each photo has its own scale, so we stretch it to 0~1 only inside that photo.
- Why the input is **518**: the ViT inside cuts the photo into 14 x 14 patches. 518 = 14 × 37.

In [ ]:
depth = core.compile_model(G / "depth-v2s.xml", DEVICE)
d = depth(normalized(frame, 518))[depth.output(0)].squeeze().astype(np.float32)
d = cv2.resize(d, (w, h))
near = (d - d.min()) / (d.max() - d.min() + 1e-8)             # 0~1 only within this photo (bigger = closer)
color = cv2.applyColorMap((near * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)   # brighter = closer
show(np.hstack([frame, color]))

In [ ]:
# Use it: portrait mode — keep only the closest 30% sharp
th = np.quantile(near, 0.7)                                   # threshold for the top 30%
keep = cv2.GaussianBlur((near >= th).astype(np.float32), (0, 0), 5)[..., None]   # soften the edge
portrait = (frame * keep + cv2.GaussianBlur(frame, (0, 0), 10) * (1 - keep)).astype(np.uint8)
show(portrait)

## Common errors
| Symptom | Cause · Fix |
|---|---|
| The mask is a mess | You did not convert to RGB, or you forgot normalization |
| `operands could not be broadcast` | You did not add a channel axis to the mask → `[..., None]` |
| The result is black or washed-out white | You did not convert 0~1 to 0~255, or you forgot `clip` |
| A gray band in the 4x result | You did not cut off the filled area |

## Summary
- Post-processing for photo→photo models: stretch to 0~1 → back to original size → to a photo (0~255, uint8).
- Normalization (mean · std) matches the input range the model learned.
- A mask or depth map is not the end result — it is **material for blending** (change background · blur · portrait mode).

## Try it
1. Replace the white background with another photo (`frame2 = grab()`, then resize it and blend).
2. Change the 0.7 in portrait mode to 0.5 or 0.9.
3. Time the three models on `"CPU"` and on `"GPU"`. Which model gains the most from the GPU?
4. Use depth to detect "the moment something comes close to the camera".
   <details><summary>Hint</summary>

   If the average of `d` (the raw value, before stretching) in the center of the photo suddenly grows, something came closer. Raw values can be compared between photos (roughly).
   </details>

---
# 🛠 Mini project — Virtual Background Camera

**What to build**: A live camera that **blurs the background or makes it one color** in the webcam video, like in a video call. Run it for 15 seconds.

**What to use**: `u2` · `normalized` (background removal from this notebook), alpha blending, `live`

**Speed tip**: The mask is made small (320 x 320) and then enlarged, so the model time stays the same even for bigger photos. But the blending (multiplying) costs more as the photo gets bigger.

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """Show the webcam video in the notebook for `seconds` seconds, changing each frame with process(photo) → photo. To stop, press I I."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # no webcam → use the photo
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n} frames · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (no webcam, so simulated with the photo)" if still is not None else ""))
    return out

In [ ]:
MODE = "blur"                 # "blur" blurred · "color" one color · "photo" another photo
BG_COLOR = (90, 160, 60)      # for one color (B, G, R)
bg_photo = None               # photo to use for "photo" — e.g. cv2.imread("beach.jpg")

def virtual_bg(f):
    h, w = f.shape[:2]
    m = u2(normalized(f, 320))[u2.output(0)].squeeze().astype(np.float32)
    m = (m - m.min()) / (m.max() - m.min() + 1e-8)
    m = cv2.resize(m, (w, h))[..., None]
    if MODE == "blur":
        bg = cv2.GaussianBlur(f, (0, 0), 15)
    elif MODE == "photo" and bg_photo is not None:
        bg = cv2.resize(bg_photo, (w, h))
    else:
        bg = np.full_like(f, BG_COLOR)
    return (f * m + bg * (1 - m)).astype(np.uint8)

live(virtual_bg, seconds=15)

**Go further**
- Raise the FPS by making the mask only every 2nd frame instead of every frame (reuse the mask)
- Use depth (section 5) to keep "only close things sharp" — blend with depth instead of the mask
- Put clock text on the background (`cv2.putText`)